# Дипломный проект — Финальный ноутбук
## Определение количества людей на видео

Финальный ноутбук содержит два способа применения обученной модели:

**Способ 1 — запуск в Colab:**
Загрузка видео → предобработка → инференс → вывод результата

**Способ 2 — Telegram бот:**
Отправка видео в бот → получение обработанного видео с боксами и статистикой

**Финальная модель:** YOLOv8m, imgsz=960
**Оптимальный порог:** conf=0.25 (по результатам Эксперимента 4)


**Pipeline разбит на функции:**
- load_model() — загрузка весов модели
- preprocess_video() — предобработка кадров
- predict_video() — инференс и постобработка
- show_result() — вывод результата
- main_pipeline() — общий запуск

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics moviepy -q
import os
import cv2
import numpy as np
from getpass import getpass
from google.colab import files
from ultralytics import YOLO
from moviepy.editor import VideoFileClip

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



In [2]:
def load_model(weights_path):
    try:
        model = YOLO(weights_path)
        print(f"Модель загружена: {weights_path}")
        return model
    except Exception as e:
        raise RuntimeError(f"Ошибка при загрузке {weights_path}: {e}")


def upload_file(default_path="people.mp4"):
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        print(f"Загружен файл: {filename}")
        return filename
    print(f"Файл не выбран, используется: {default_path}")
    return default_path


def preprocess_video(input_path):
    cap = cv2.VideoCapture(input_path)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    output_path = "preprocessed.mp4"
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        l = clahe.apply(l)
        lab = cv2.merge((l, a, b))
        frame = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

        out.write(frame)

    cap.release()
    out.release()
    print("Предобработка завершена")
    return output_path


def predict_video(model, input_path, output_path="output.mp4", conf=0.25):
    cap = cv2.VideoCapture(input_path)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    counts_per_frame = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model.track(
            frame, conf=conf, persist=True,
            tracker="bytetrack.yaml", verbose=False
        )

        count = 0
        if results[0].boxes.id is not None:
            for box in results[0].boxes:
                if int(box.cls[0]) == 0:
                    count += 1

        counts_per_frame.append(count)

        annotated_frame = results[0].plot()
        cv2.putText(
            annotated_frame, f"People: {count}",
            (20, 40), cv2.FONT_HERSHEY_SIMPLEX,
            1, (0, 255, 0), 2
        )
        out.write(annotated_frame)

    cap.release()
    out.release()

    print(f"\nСтатистика по видео:")
    print(f"Кадров обработано:   {len(counts_per_frame)}")
    print(f"Макс. людей в кадре: {max(counts_per_frame)}")
    print(f"Мин. людей в кадре:  {min(counts_per_frame)}")
    print(f"Среднее в кадре:     {np.mean(counts_per_frame):.1f}")

    return output_path


def show_result(output_path="output.mp4"):
    clip = VideoFileClip(output_path)
    return clip.ipython_display(width=640, maxduration=150)


def main_pipeline(weights_path, conf=0.25):
    input_path = upload_file()
    clean_path = preprocess_video(input_path)
    model = load_model(weights_path)
    processed_video = predict_video(model, clean_path, conf=conf)
    return show_result(processed_video)

In [ ]:
WEIGHTS_PATH = '/content/drive/MyDrive/yolo_training/SecondUpdatedModel/weights/last.pt'
main_pipeline(WEIGHTS_PATH)

## Telegram бот

Бот принимает видеофайл от пользователя, прогоняет его через ту же
модель и pipeline (включая трекинг ByteTrack для стабильности боксов),
и отправляет обработанное видео обратно.

Используется единая функция predict_video() из основного pipeline —
дублирования логики обработки нет.

**Параметры:**
- Модель: YOLOv8m, imgsz=960
- conf: 0.25 (оптимальный порог по результатам Эксперимента 4)
- Токен бота запрашивается через getpass — не хранится в коде

**Команды бота:**
- /start — приветствие и информация о модели
- Отправка видео — запускает обработку и возвращает результат

In [3]:
!pip install python-telegram-bot nest_asyncio -q
!pip install lap -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.4/769.4 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.0 MB/s eta 0:00:00


In [ ]:
import nest_asyncio
from getpass import getpass
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, ContextTypes, filters

nest_asyncio.apply()

TOKEN = getpass("Введите токен бота: ")
WEIGHTS_PATH = '/content/drive/MyDrive/yolo_training/SecondUpdatedModel/weights/last.pt'
CONF_THRESHOLD = 0.25

model = YOLO(WEIGHTS_PATH)


async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "Приветствую. Модель: YOLOv8m (imgsz=960).\n"
        "Отправьте видеофайл для подсчёта людей."
    )


async def handle_video(update: Update, context: ContextTypes.DEFAULT_TYPE):
    video = update.message.video or update.message.document
    if not video:
        await update.message.reply_text("Пожалуйста, отправьте видеофайл")
        return

    await update.message.reply_text("Обрабатываю видео, это может занять время...")

    input_path = "input.mp4"
    output_path = "output.mp4"
    compressed_path = "output_compressed.mp4"

    file = await video.get_file()
    await file.download_to_drive(input_path)

    predict_video(model, input_path, output_path=output_path, conf=CONF_THRESHOLD)

    # Сжатие для соответствия лимиту Telegram (50 МБ)
    os.system(
        f'ffmpeg -y -i {output_path} -vcodec libx264 -crf 28 '
        f'-preset fast {compressed_path}'
    )

    with open(compressed_path, "rb") as f:
        await update.message.reply_video(f)

    os.remove(input_path)
    os.remove(output_path)
    os.remove(compressed_path)

await app.run_polling()